В данном ноутбуке мы будем получать через скрепинг вакансии из ххру) 

In [22]:
%pip install -U selenium

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from time import sleep
import random

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options

In [40]:
options = Options()
options.page_load_strategy = 'eager'
driver = webdriver.Chrome(options=options)

In [ ]:
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true')

In [ ]:
def safeGetElement(element, selector):
    try:
        return element.find_element(By.CSS_SELECTOR, selector)
    except:
        return None

def safeExtractText(element, selector):
    try:
        return  getattr(element.find_element(By.CSS_SELECTOR, selector), 'text', '')
    except:
        return ''

def safeGetAttr(element, attr):
    try:
        return element.get_attribute(attr)
    except:
        return ''

def randSleep():
    sleep(random.uniform(1, 7))


In [ ]:
df = pd.DataFrame({'title': [],'link': [], 'address': [], 'company': [], 'short_description': [], 'rich_description': [], 'salary': []})

In [27]:
def collect_vacancies():
    page_df = pd.DataFrame({'title': [],'link': [], 'address': [], 'company': [], 'short_description': []})
    vacs = driver.find_elements(By.CSS_SELECTOR, "[data-qa=vacancy-serp__vacancy]")

    for vac in vacs:
        link = safeGetAttr(safeGetElement(vac, "[data-qa=serp-item__title]"), 'href')

        duplicates = page_df.loc[page_df['link'] == link]

        if len(duplicates) > 0:
            continue

        title = safeExtractText(vac, "[data-qa=serp-item__title-text]")
        address = safeExtractText(vac, "[data-qa=vacancy-serp__vacancy-address]")
        company = safeExtractText(vac, "[data-qa=vacancy-serp__vacancy-employer-text]")
        short_description_container = safeGetElement(vac, "[data-qa=vacancy-serp__vacancy_snippet_requirement]")

        short_description = ''
        if short_description_container != '':
            short_description = safeExtractText(short_description_container, "span")
        
        new_row = pd.DataFrame({'title': [title], 'link': [link], 'address': [address], 'company': [company], 'short_description': [short_description]})
        page_df = pd.concat([page_df, new_row], ignore_index=True)

    return page_df

In [ ]:
next = safeGetElement(driver, "[data-qa=pager-next]")
counter = 0

while next and counter <= 2: 
    randSleep()
    page_df = collect_vacancies()
    df = pd.concat([df, page_df], ignore_index=True)

    if next:
        ActionChains(driver).move_to_element(next).perform()
        ActionChains(driver).scroll_by_amount(0, 500).perform()

        next.click()
        counter += 1
        next = safeGetElement(driver, "[data-qa=pager-next]")

                                                title  \
0                                      Бухгалтер в IT   
1                              IT-рекрутер в компанию   
2                     Преподаватель Python в IT-школу   
3       Менеджер по работе с клиентами (IT-LegalTech)   
4                                    Грузчик на склад   
5                                          Мобилограф   
6                                       IT специалист   
7                                          IT-инженер   
8                                          IT Аудитор   
9                                         IT Директор   
10                                        IT менеджер   
11                                      IT-специалист   
12                                      IT специалист   
13                                      IT-специалист   
14                                         IT Инженер   
15                                         IT инженер   
16                             

In [29]:
df

,title,link,address,company,short_description
0,Бухгалтер в IT,https://hh.ru/vacancy/131054157?query=it&hhtmF...,"Минск, улица Петра Мстиславца, 9",ООО Спортдата,Высшее образование. Опыт работы в
1,IT-рекрутер в компанию,https://hh.ru/vacancy/131053198?query=it&hhtmF...,Астана,ТОО Smart Commerce Kazakhstan,...если в
2,Преподаватель Python в IT-школу,https://hh.ru/vacancy/131072601?query=it&hhtmF...,Алматы,ТОО ACADEMICA COURSES,Опыт работы Middle/Senior Data Analyst/Data Sc...
3,Менеджер по работе с клиентами (IT-LegalTech),https://hh.ru/vacancy/131443599?query=it&hhtmF...,"Ташкент, улица Тахтапуль Дарвоза, 328",ООО HESAP,"Обучаем: Даже если нет опыта, мы сделаем из те..."
4,Грузчик на склад,https://hh.ru/vacancy/131495477?query=it&hhtmF...,Москва,NetLab,
...,...,...,...,...,...
145,Менеджер по продажам IT-решений,https://hh.ru/vacancy/124067425?query=it&hhtmF...,"Волгоград, р-н Центральный",Интегрикс,Понимание трендов
146,"Юрист в IT-компанию (финтех, IT Park, АО)",https://hh.ru/vacancy/131107981?query=it&hhtmF...,"Ташкент, 2-й проезд Миробод, 39",АО Global Solutions,"Опыт работы: от 3 лет, желательно в"
147,Специалист по IT-поддержке и цифровой инфрастр...,https://hh.ru/vacancy/131927764?query=it&hhtmF...,Москва,ООО Селлерпро,Уверенная работа с Google Docs и Excel. Опыт н...
148,Менеджер по продажам IT-сервис,https://hh.ru/vacancy/131909239?query=it&hhtmF...,Астрахань,ООО Джусап,Опыт в продажах от 1 года. Умение делать от 90...


## Парсинг конкретной вакансии

По уже заготовленному датасету ходим по каждой вакансии и получаем детализированное описание и зарплату. Текст вакансии мы конвертируем из html в md через html-to-markdown

In [30]:
%pip install html-to-markdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 6.8 MB/s eta 0:00:006.8 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [31]:
from html_to_markdown import convert

In [ ]:
def parse_vac(link):
    driver.get(link, )
    descriptionHTML = safeGetElement(driver, "[data-qa=vacancy-description]")
    description = ''

    if descriptionHTML:
        description = convert(descriptionHTML.get_attribute('outerHTML'))

    salary = safeExtractText(driver, "[data-qa=vacancy-salary-compensation-type-gross]")
    print(description, salary)

In [41]:
parse_vac('https://hh.ru/vacancy/131053198')

{'content': 'Ищем IT-рекрутера в POSCENTER (Астана)\n\n\n\nХочешь закрывать интересные вакансии, быстро расти и работать в стабильной компании?\n\n\n\nПрисоединяйся к нам! Мы ищем рекрутера, который будет искать и закрывать специалистов для нашей команды.\n\n\n\n**Что ты будешь делать:**\n\n\n\n- Полный цикл подбора: от поиска до выхода человека на работу\n- Искать разработчиков, менеджеров по продажам и сервисных инженеров\n- Размещать вакансии на hh.kz, LinkedIn, Telegram, Instagram и других площадках\n- Активно искать людей в LinkedIn, Telegram, GitHub и чатах\n- Проводить звонки и интервью, координировать встречи с руководителями\n- Вести кандидатов в CRM и давать обратную связь\n- Помогать новым сотрудникам адаптироваться\n\n\n**Кого мы ищем:**\n\n\n\n- Опыт работы рекрутером от 1 года (будет здорово, если в IT, digital или retail)\n- Умеешь хорошо искать кандидатов на hh.kz и LinkedIn\n- Понимаешь хотя основы IT-специальностей\n- Общительный, энергичный и стрессоустойчивый\n- Уме